In [1]:
from ome_zarr.image import NgffImage, NgffMultiscales
from ome_zarr.scene import NgffScene
from ome_zarr_models._v06.coordinate_transforms import CoordinateSystem, CoordinateSystemIdentifier, Axis, Translation
from bioio import BioImage
import os
import numpy as np
from dask.distributed import Client, get_client
import napari
import xmltodict

from multiview_stitcher import spatial_image_utils as si_utils
from multiview_stitcher import fusion
from skimage import io
import glob
from pathlib import Path

In [2]:
viewer = napari.Viewer()

In [3]:
try:
  client = get_client()
except ValueError:
  client = Client()

print(client.dashboard_link)

http://127.0.0.1:8787/status


In [4]:
root = r'E:\UserData\Johannes\20260420\2026_04_20_14_08_41--TRG1970_002\TileScan 1'
file_names = glob.glob(os.path.join(root, "**", "*ome.tif"), recursive=True)
file_names

['E:\\UserData\\Johannes\\20260420\\2026_04_20_14_08_41--TRG1970_002\\TileScan 1\\C\\5.ome.tif',
 'E:\\UserData\\Johannes\\20260420\\2026_04_20_14_08_41--TRG1970_002\\TileScan 1\\D\\3.ome.tif']

In [5]:
os.path.join(root, "**", "*ome.tif")

'E:\\UserData\\Johannes\\20260420\\2026_04_20_14_08_41--TRG1970_002\\TileScan 1\\**\\*ome.tif'

In [6]:
def find_in_nested_dict(data, key, value):
    """
    Recursively search for objects containing a specific key-value pair.
    Returns a list of all matching objects.
    """
    results = []
    
    def search(obj):
        if isinstance(obj, dict):
            if obj.get(key) == value:
                results.append(obj)
            for v in obj.values():
                search(v)
        elif isinstance(obj, list):
            for item in obj:
                search(item)
    
    search(data)
    return results  

In [7]:
write_jobs = []
images = []
transforms = []
for filename in file_names:
	img_basename = Path(filename).stem.split(".")[0] 
	outfile_oz = img_basename + ".ome.zarr"
	bf = BioImage(filename)
	

	if bf.ome_metadata.images[0].pixels.physical_size_z is None:
		size_z = 0
	else:
		size_z = bf.ome_metadata.images[0].pixels.physical_size_z *1e6

	scale = {
		"z": size_z,
		"y": bf.ome_metadata.images[0].pixels.physical_size_y *1e6,
		"x": bf.ome_metadata.images[0].pixels.physical_size_x *1e6
	}

	axes_units = {
		"z": "micrometer",
		"y": "micrometer",
		"x": "micrometer",
	}
	
	# parse tile metadata
	metadata_file = os.path.join(Path(filename).parent, "Metadata", f"{img_basename}.xlif")
	metadata_dict = xmltodict.parse(open(metadata_file).read())
	tile_scan_info = find_in_nested_dict(metadata_dict, '@Name', 'TileScanInfo')[0]["Tile"]

	# n_series= bf.series_count()
	n_series = len(bf.scenes)

	if n_series > 1:
		sims = []
		for idx in range(n_series):
			bf.set_scene(bf.scenes[idx])
			dask_image = bf.dask_data
			# dask_image = bf.to_dask(series=idx)
			ts_info = tile_scan_info[idx]

			t_z = float(ts_info["@PosZ"]) * 1e6
			t_y = float(ts_info["@PosY"]) * 1e6
			t_x = float(ts_info["@PosX"]) * 1e6

			sims.append(
				si_utils.get_sim_from_array(
					dask_image,
					dims=["t", "c", "z", "y", "x"],
					scale=scale,
					translation={"z": t_z, "y": t_y, "x": t_x},
					transform_key="stage_metadata",
					)
			)

		image = fusion.fuse(sims, transform_key="stage_metadata", fusion_func=fusion.max_fusion)
	else:
		# image = bf.to_dask(series=0)
		image = bf.dask_data
		t_z = float(tile_scan_info["@PosZ"]) * 1e6
		t_y = float(tile_scan_info["@PosY"]) * 1e6
		t_x = float(tile_scan_info["@PosX"]) * 1e6

	ngff_image = NgffImage(
		image.squeeze(),
		dims=["z", "y", "x"],
		scale=scale,
		axes_units=axes_units,
		name=str(img_basename)
	)
	ngff_multiscales = NgffMultiscales(
		image=ngff_image,
		scale_factors=[
		{"z": 2, "y": 2, "x": 2},
		{"z": 4, "y": 4, "x": 4},
		{"z": 8, "y": 8, "x": 8},
		{"z": 16, "y": 16, "x": 16},
		],
	)
	images.append(ngff_multiscales)

	transform = Translation(
		translation=(t_z, t_y, t_x),
		input=CoordinateSystemIdentifier(
			path=ngff_multiscales.name,
			name=ngff_multiscales.metadata.coordinateSystems[0].name
		),
		output=CoordinateSystemIdentifier(
			name="world"
		)
	)
	transforms.append(transform)

<string>:5: UserWarning: Dimension z is too small to downsample further.
<string>:5: UserWarning: Dimension z is too small to downsample further.


In [8]:

cs_world = CoordinateSystem(
	name="world",
	axes=(
		Axis(name="z", type="space", unit="micrometer"),
		Axis(name="y", type="space", unit="micrometer"),
		Axis(name="x", type="space", unit="micrometer"),
	)
)

ngff_scene = NgffScene(
	images=images,
	transformations=transforms,
	coordinate_systems=(
		cs_world,
	)
)

delayed_objects = ngff_scene.to_ome_zarr(
	os.path.join(root, "scene.ome.zarr")
)

C:\Users\Leica-User\Documents\GitHub\ome-zarr-py\ome_zarr\writer.py:869: FutureWarning: Passing storage-related arguments via **kwargs is deprecated. Please use the 'zarr_store_kwargs' parameter instead. **kwargs will be removed in a future version.
  da.to_zarr(


## Show scenes



In [9]:
viewer.open(os.path.join(root, "scene.ome.zarr"), plugin="napari-ome-zarr")

Root group {'ome': {'scene': {'coordinateTransformations': [{'type': 'translation', 'input': {'name': 'physical', 'path': '5'}, 'output': {'name': 'world'}, 'translation': [2887.5623, 29521.6174, 49806.7578]}, {'type': 'translation', 'input': {'name': 'physical', 'path': '3'}, 'output': {'name': 'world'}, 'translation': [2887.5623, 38521.6192, 31806.756899999997]}], 'coordinateSystems': [{'name': 'world', 'axes': [{'name': 'z', 'type': 'space', 'unit': 'micrometer'}, {'name': 'y', 'type': 'space', 'unit': 'micrometer'}, {'name': 'x', 'type': 'space', 'unit': 'micrometer'}]}]}, 'version': '0.6'}}
Scene.matches {'scene': {'coordinateTransformations': [{'type': 'translation', 'input': {'name': 'physical', 'path': '5'}, 'output': {'name': 'world'}, 'translation': [2887.5623, 29521.6174, 49806.7578]}, {'type': 'translation', 'input': {'name': 'physical', 'path': '3'}, 'output': {'name': 'world'}, 'translation': [2887.5623, 38521.6192, 31806.756899999997]}], 'coordinateSystems': [{'name': 'w

[<Image layer 'scene.ome' at 0x1388341aea0>,
 <Image layer 'scene.ome [1]' at 0x138895e7ad0>]

In [18]:
layer = viewer.layers['scene.ome [1]']
max_projection = layer.data[0].min(axis=0)
viewer.add_image(max_projection, scale = layer.affine.scale[1:], translate=layer.affine.translate[1:], name="max projection")

<Image layer 'max projection [1]' at 0x14656604890>

In [ ]:
for layer in viewer.layers:
	projection = layer.data[0].min(axis=0).rechunk(640, 640)

	# iterate over chunks and save as individual images
	for idx in np.ndindex(projection.numblocks):
		chunk = projection.blocks[idx]
		if not chunk.shape == (640, 640):
			continue

		client.submit(io.imsave, os.path.join(r"C:\Users\johan\Desktop\local_data\BiaPoL\autoSCAPE\20260317\projections", f"{layer.name}_chunk_{idx}.tif"), chunk)

In [46]:
viewer_projections.layers.clear()
viewer_projections.add_image(chunk, contrast_limits=(p5.compute(), p95.compute()))

<Image layer 'chunk' at 0x1d8aa80ea20>

In [ ]:
viewer.layers[0].data[0].min(axis=0).rechunk(640, 640)

(4, 5)

In [35]:
p5.compute()

np.float64(243.0)

In [36]:
p95.compute()

np.float64(981.0)